In [0]:
dbutils.widgets.text("batch_id","")
v_batch_id = dbutils.widgets.get("batch_id")

In [0]:
%run ../0-common/env-config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.drivers"

silver_table = f"{catalog_name}.{silver_schema}.drivers"

In [0]:
drivers_df = spark.read.table(bronze_table)

In [0]:
from pyspark.sql import functions as F

In [0]:
drivers_selected_df = (
    spark.read.table(bronze_table).filter(F.col("batch_id") == v_batch_id)
    )

In [0]:
drivers_renamed_df = (
    drivers_selected_df
    .withColumnsRenamed({
        "driverId":"driver_id",
        "dateOfBirth":"date_of_birth"
    })
)

In [0]:
drivers_concat_df = (
    drivers_renamed_df
    .withColumn("driver_name",
                 F.concat_ws(" ", F.initcap("name.givenName"),
                              F.initcap("name.familyName")
                            ))
    .drop(F.col("name"))
)

In [0]:
drivers_distinct_df = drivers_concat_df.dropDuplicates(["driver_id"])

In [0]:
drivers_final_df = (
    drivers_distinct_df
    .withColumn("nationality", F.initcap("nationality"))
    .withColumn("created_at", F.current_timestamp())
    .withColumn("updated_at", F.current_timestamp())
)

In [0]:
if not spark.catalog.tableExists(silver_table):
    (
        drivers_final_df.write
        .format('delta')
        .mode("overwrite")
        .saveAsTable(silver_table)
    )
else:
    from delta.tables import DeltaTable

    delta_table = DeltaTable.forName(spark, silver_table)
    (
        delta_table.alias("t")
        .merge(
            drivers_final_df.alias("d"),
            "t.driver_id = d.driver_id"
        )
        .whenMatchedUpdate(
            condition="d.batch_id >= t.batch_id",
            set={
                "date_of_birth": "d.date_of_birth",
                "nationality": "d.nationality",
                "url": "d.url",
                "ingestion_timestamp": "d.ingestion_timestamp",
                "source_file": "d.source_file",
                "batch_id": "d.batch_id",
                "driver_name": "d.driver_name",
                "updated_at": "d.updated_at"
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )